# TCGplayer Inventory Reconciliation Reconstruction

## Objective

Reconstruct the historical trading-card inventory reconciliation workflow using Python and pandas.

The original workflow used spreadsheet-based matching and business logic to combine TCGplayer inventory data with Trader Tools market data.

This notebook will:

1. Load the paired historical source datasets.
2. Inspect and profile their structure.
3. Normalize fields used for matching.
4. Reconstruct the historical Name + Edition matching logic.
5. Identify unmatched and ambiguous records.
6. Compare Python results against the historical Processing output.
7. Document opportunities to improve reliability beyond the original workbook.

## 1. Setup and Data Inspection

Load the reconstruction inputs and the preserved historical Processing reference, verify the expected files are available, and inspect the basic structure of each dataset before applying matching logic.


In [1]:
import pandas as pd
from pathlib import Path

project_root = Path("..")

In [2]:
tcgplayer_file = project_root / "data" / "reconstruction_inputs" / "tcgplayer_inventory_input.csv"
trader_tools_file = project_root / "data" / "reconstruction_inputs" / "trader_tools_market_input.csv"
historical_reference_file = project_root / "data" / "reference_outputs" / "historical_processing_reference.csv"

print(tcgplayer_file.exists())
print(trader_tools_file.exists())
print(historical_reference_file.exists())

True
True
True


In [3]:
tcgplayer = pd.read_csv(tcgplayer_file)
trader_tools = pd.read_csv(trader_tools_file)
historical_reference = pd.read_csv(historical_reference_file)

In [4]:
print("TCGplayer:", tcgplayer.shape)
print("Trader Tools:", trader_tools.shape)
print("Historical Reference:", historical_reference.shape)

TCGplayer: (4368, 17)
Trader Tools: (4062, 11)
Historical Reference: (4499, 6)


In [5]:
display(tcgplayer.head())
display(trader_tools.head())
display(historical_reference.head())

,Count,Tradelist Count,Name,Edition,Card Number,Condition,Language,Foil,Signed,Artist Proof,Altered Art,Misprint,Promo,Textless,My Price,Rarity,Price
0,1,0,Abbey Matron,Homelands,103,Near Mint,English,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Common,$0.18
1,24,8,Abbot of Keral Keep,Magic Origins,127,Near Mint,English,NaN,NaN,NaN,NaN,NaN,NaN,NaN,175.0,Rare,$2.37
2,64,8,Abhorrent Overlord,Theros,75,Near Mint,English,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25.0,Rare,$0.26
3,1,0,Abolish,Prophecy,1,Near Mint,English,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Uncommon,$0.24
4,2,0,Abomination,Legends,1,Near Mint,English,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Uncommon,$0.45


,cardName,setName,foil,qtyOwned,buyPrice,sellPrice,buyMerchant,sellMerchant,buyQty,sellQty,spread
0,NaN,NaN,0,NaN,NaN,0.00,NaN,NaN,NaN,0.0,%
1,Dragon Broodmother,Alara Reborn,0,1.0,10.51,14.02,isleofcards,isleofcards,10.0,2.0,%
2,"Karrthus, Tyrant of Jund",Alara Reborn,0,1.0,11.06,14.28,isleofcards,isleofcards,11.0,1.0,%
3,Lord of Extinction,Alara Reborn,0,1.0,11.31,16.17,isleofcards,isleofcards,4.0,6.0,%
4,Sen Triplets,Alara Reborn,0,1.0,17.44,23.73,isleofcards,isleofcards,8.0,4.0,%


,Count,Tradelist Count,Name,Edition,My Price,Price
0,1,1,Abbey Matron,Homelands,0.18,$0.18
1,24,8,Abbot of Keral Keep,Magic Origins,1.63,$2.37
2,64,8,Abhorrent Overlord,Theros,0.28,$0.26
3,1,1,Abolish,Prophecy,0.24,$0.24
4,2,2,Abomination,Legends,0.43,$0.45


In [6]:
print("TCGplayer missing Name:", tcgplayer["Name"].isna().sum())
print("TCGplayer missing Edition:", tcgplayer["Edition"].isna().sum())

print("Trader Tools missing cardName:", trader_tools["cardName"].isna().sum())
print("Trader Tools missing setName:", trader_tools["setName"].isna().sum())

print(
    "TCGplayer duplicate Name + Edition rows:",
    tcgplayer.duplicated(subset=["Name", "Edition"], keep=False).sum()
)

print(
    "Trader Tools duplicate cardName + setName rows:",
    trader_tools.duplicated(subset=["cardName", "setName"], keep=False).sum()
)

TCGplayer missing Name: 0
TCGplayer missing Edition: 0
Trader Tools missing cardName: 1
Trader Tools missing setName: 1
TCGplayer duplicate Name + Edition rows: 420
Trader Tools duplicate cardName + setName rows: 0


## 2. Duplicate Record and Key Analysis

Before reconstructing the historical matching logic, evaluate repeated `Name + Edition` combinations to determine whether they represent data-quality problems or legitimate inventory distinctions.

The goal is to distinguish the **inventory record identity** from the simpler key used to connect inventory records to external market data.


In [7]:
duplicate_rows = (
    tcgplayer[
        tcgplayer.duplicated(
            subset=["Name", "Edition"],
            keep=False
        )
    ]
    .sort_values(["Name", "Edition"])
)

print(
    "Duplicate Name + Edition groups:",
    duplicate_rows.groupby(["Name", "Edition"]).ngroups
)

display(
    duplicate_rows[
        [
            "Name",
            "Edition",
            "Card Number",
            "Condition",
            "Language",
            "Foil",
            "My Price",
            "Price"
        ]
    ].head(25)
)

Duplicate Name + Edition groups: 172


,Name,Edition,Card Number,Condition,Language,Foil,My Price,Price
79,Amnesia,The Dark,19,Good (Lightly Played),English,NaN,NaN,$0.64
80,Amnesia,The Dark,19,Near Mint,English,NaN,NaN,$0.64
166,Argothian Treefolk,Antiquities,60,Played,English,NaN,NaN,$0.20
167,Argothian Treefolk,Antiquities,60,Near Mint,English,NaN,NaN,$0.20
176,Armor Thrull,Fallen Empires,1,Near Mint,English,NaN,NaN,$0.14
177,Armor Thrull,Fallen Empires,2,Near Mint,English,NaN,NaN,$0.17
178,Armor Thrull,Fallen Empires,3,Near Mint,English,NaN,NaN,$0.17
179,Armor Thrull,Fallen Empires,4,Near Mint,English,NaN,NaN,$0.17
188,Artifact Possession,Antiquities,45,Good (Lightly Played),English,NaN,NaN,$0.19
189,Artifact Possession,Antiquities,45,Near Mint,English,NaN,NaN,$0.19


In [8]:
duplicate_analysis = (
    tcgplayer
    .groupby(["Name", "Edition"])
    .agg(
        inventory_rows=("Name", "size"),
        card_numbers=("Card Number", "nunique"),
        conditions=("Condition", "nunique"),
        languages=("Language", "nunique")
    )
    .reset_index()
)

duplicate_analysis = duplicate_analysis[
    duplicate_analysis["inventory_rows"] > 1
]

print("Duplicate Name + Edition groups:", len(duplicate_analysis))

print(
    "Groups with multiple card numbers:",
    (duplicate_analysis["card_numbers"] > 1).sum()
)

print(
    "Groups with multiple conditions:",
    (duplicate_analysis["conditions"] > 1).sum()
)

display(
    duplicate_analysis.sort_values(
        ["card_numbers", "inventory_rows"],
        ascending=False
    ).head(20)
)

Duplicate Name + Edition groups: 172
Groups with multiple card numbers: 46
Groups with multiple conditions: 118


,Name,Edition,inventory_rows,card_numbers,conditions,languages
2221,Mishra's Factory,Antiquities,5,4,3,1
172,Armor Thrull,Fallen Empires,4,4,1,1
264,Basal Thrull,Fallen Empires,4,4,1,1
407,Brassclaw Orcs,Fallen Empires,4,4,1,1
623,Combat Medic,Fallen Empires,4,4,1,1
1049,Elven Fortress,Fallen Empires,4,4,1,1
1481,Goblin War Drums,Fallen Empires,4,4,1,1
1662,Homarid,Fallen Empires,4,4,1,1
1710,Hymn to Tourach,Fallen Empires,4,4,1,1
1717,Icatian Infantry,Fallen Empires,4,4,1,1


In [9]:
duplicate_analysis["multiple_card_numbers"] = duplicate_analysis["card_numbers"] > 1
duplicate_analysis["multiple_conditions"] = duplicate_analysis["conditions"] > 1

print("Card number only:",
      ((duplicate_analysis["multiple_card_numbers"]) &
       (~duplicate_analysis["multiple_conditions"])).sum())

print("Condition only:",
      ((~duplicate_analysis["multiple_card_numbers"]) &
       (duplicate_analysis["multiple_conditions"])).sum())

print("Both:",
      ((duplicate_analysis["multiple_card_numbers"]) &
       (duplicate_analysis["multiple_conditions"])).sum())

print("Neither:",
      ((~duplicate_analysis["multiple_card_numbers"]) &
       (~duplicate_analysis["multiple_conditions"])).sum())

Card number only: 41
Condition only: 113
Both: 5
Neither: 13


In [10]:
neither_groups = duplicate_analysis[
    (~duplicate_analysis["multiple_card_numbers"]) &
    (~duplicate_analysis["multiple_conditions"])
][["Name", "Edition"]]

neither_rows = tcgplayer.merge(
    neither_groups,
    on=["Name", "Edition"],
    how="inner"
)

display(
    neither_rows[
        [
            "Name",
            "Edition",
            "Card Number",
            "Condition",
            "Language",
            "Foil",
            "Signed",
            "Artist Proof",
            "Altered Art",
            "Misprint",
            "Promo",
            "Textless",
            "My Price",
            "Price"
        ]
    ].sort_values(["Name", "Edition"])
)

,Name,Edition,Card Number,Condition,Language,Foil,Signed,Artist Proof,Altered Art,Misprint,Promo,Textless,My Price,Price
0,Blood Moon,The Dark,57,Good (Lightly Played),English,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,$59.00
1,Blood Moon,The Dark,57,Good (Lightly Played),English,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,$59.00
2,Eater of the Dead,The Dark,6,Near Mint,English,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,$5.00
3,Eater of the Dead,The Dark,6,Near Mint,English,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,$5.00
4,Fountain of Youth,The Dark,100,Near Mint,English,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,$0.29
5,Fountain of Youth,The Dark,100,Near Mint,English,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,$0.29
6,"Kytheon, Hero of Akros",Magic Origins,23,Near Mint,Japanese,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1150.0,$10.02
7,"Kytheon, Hero of Akros",Magic Origins,23,Near Mint,Korean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1250.0,$10.02
8,Lightning Bolt,Revised Edition,161,Near Mint,English,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,$2.46
9,Lightning Bolt,Revised Edition,161,Near Mint,English,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,$2.46


In [11]:
identity_fields = [
    "Name",
    "Edition",
    "Card Number",
    "Condition",
    "Language",
    "Foil",
    "Signed",
    "Artist Proof",
    "Altered Art",
    "Misprint",
    "Promo",
    "Textless"
]

display(
    neither_rows[
        [
            "Count",
            "Tradelist Count",
            *identity_fields,
            "My Price",
            "Price"
        ]
    ].sort_values(["Name", "Edition", "Language"])
)

,Count,Tradelist Count,Name,Edition,Card Number,Condition,Language,Foil,Signed,Artist Proof,Altered Art,Misprint,Promo,Textless,My Price,Price
0,2,0,Blood Moon,The Dark,57,Good (Lightly Played),English,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,$59.00
1,1,0,Blood Moon,The Dark,57,Good (Lightly Played),English,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,$59.00
2,1,0,Eater of the Dead,The Dark,6,Near Mint,English,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,$5.00
3,1,0,Eater of the Dead,The Dark,6,Near Mint,English,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,$5.00
4,2,0,Fountain of Youth,The Dark,100,Near Mint,English,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,$0.29
5,1,0,Fountain of Youth,The Dark,100,Near Mint,English,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,$0.29
6,1,1,"Kytheon, Hero of Akros",Magic Origins,23,Near Mint,Japanese,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1150.0,$10.02
7,1,1,"Kytheon, Hero of Akros",Magic Origins,23,Near Mint,Korean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1250.0,$10.02
8,4,0,Lightning Bolt,Revised Edition,161,Near Mint,English,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,$2.46
9,2,0,Lightning Bolt,Revised Edition,161,Near Mint,English,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,$2.46


### Duplicate Analysis Findings

The analysis identified:

- 172 repeated `Name + Edition` groups
- 41 groups explained by multiple card numbers
- 113 groups explained by multiple conditions
- 5 groups containing both multiple card numbers and multiple conditions
- 13 groups requiring additional review

Further inspection showed that the remaining groups were primarily caused by language variants or multiple inventory rows with the same identifying attributes but different quantities.

These records are not automatically considered data-quality errors. They reflect legitimate inventory distinctions such as artwork/card-number variants, condition, language, and separately recorded quantities.

This establishes an important distinction:

**Inventory record identity** may depend on:

`Name + Edition + Card Number + Condition + Language + variant attributes`

while the historical external-pricing match appears to have operated primarily on:

`Name + Edition`

For the historical reconstruction, original inventory rows are preserved rather than aggregated or corrected. Potential improvements to identity and aggregation belong to the later modernization analysis.


## 3. Historical Matching Reconstruction

The historical workbook appears to have matched TCGplayer inventory records to Trader Tools market data using card name and edition.

To reconstruct that behavior, normalized `Name + Edition` fields are created while preserving the original source values.

Because multiple TCGplayer inventory records may legitimately share the same card name and edition, the expected relationship is **many inventory records to one market-data record**.


In [12]:
# Create normalized matching fields without modifying the original columns

tcgplayer["match_name"] = (
    tcgplayer["Name"]
    .str.strip()
    .str.lower()
)

tcgplayer["match_edition"] = (
    tcgplayer["Edition"]
    .str.strip()
    .str.lower()
)

trader_tools["match_name"] = (
    trader_tools["cardName"]
    .str.strip()
    .str.lower()
)

trader_tools["match_edition"] = (
    trader_tools["setName"]
    .str.strip()
    .str.lower()
)

In [13]:
print(
    tcgplayer[
        ["Name", "Edition", "match_name", "match_edition"]
    ].head()
)

print()

print(
    trader_tools[
        ["cardName", "setName", "match_name", "match_edition"]
    ].head()
)

                  Name        Edition           match_name  match_edition
0         Abbey Matron      Homelands         abbey matron      homelands
1  Abbot of Keral Keep  Magic Origins  abbot of keral keep  magic origins
2   Abhorrent Overlord         Theros   abhorrent overlord         theros
3              Abolish       Prophecy              abolish       prophecy
4          Abomination        Legends          abomination        legends

                   cardName       setName                match_name  \
0                       NaN           NaN                       NaN   
1        Dragon Broodmother  Alara Reborn        dragon broodmother   
2  Karrthus, Tyrant of Jund  Alara Reborn  karrthus, tyrant of jund   
3        Lord of Extinction  Alara Reborn        lord of extinction   
4              Sen Triplets  Alara Reborn              sen triplets   

  match_edition  
0           NaN  
1  alara reborn  
2  alara reborn  
3  alara reborn  
4  alara reborn  


In [14]:
reconciliation = tcgplayer.merge(
    trader_tools[
        [
            "match_name",
            "match_edition",
            "buyPrice",
            "sellPrice"
        ]
    ],
    on=["match_name", "match_edition"],
    how="left",
    validate="many_to_one",
    indicator=True
)

print(reconciliation["_merge"].value_counts())

_merge
both          4254
left_only      114
right_only       0
Name: count, dtype: int64


## Initial Reconciliation Results

The normalized `Name + Edition` merge successfully validated as a many-to-one relationship between TCGplayer inventory and Trader Tools market data.

Initial results:

- 4,254 TCGplayer inventory rows matched a Trader Tools market record.
- 114 TCGplayer inventory rows did not find a corresponding market-data match.
- The many-to-one validation completed without error, confirming that the Trader Tools matching key is unique for the records used in the reconciliation.

The unmatched records will be reviewed separately to determine whether they represent naming differences, edition differences, missing market data, or other reconciliation exceptions.

### Historical Source-Pair Provenance

The reconstruction inputs used here were preserved within the same historical workbook state: the TCGplayer-side inventory input and the `TT Data` market-data worksheet.

That supports using them together for reconstruction, but it does **not** independently prove that a surviving standalone Trader Tools export is the exact temporal export originally imported into the workbook.

Accordingly, this notebook treats the preserved workbook data as the reconstruction basis and avoids making a stronger file-provenance claim until it is independently validated.

> **Important:** No claim is made here that historical `My Price` should equal Trader Tools `sellPrice`. The historical workbook contains additional pricing/business logic, so provenance must be validated separately from price equality.


## 4. Unmatched Record Investigation

The initial reconciliation produced 114 unmatched TCGplayer inventory rows.

Rather than automatically modifying the matching logic, these records are treated as reconciliation exceptions and investigated to determine whether the cause is:

- a missing card in the Trader Tools dataset;
- a set/edition naming difference;
- a card-name formatting difference; or
- another source-system discrepancy.

This preserves the original data while allowing matching improvements to be based on observed evidence rather than assumptions.


In [15]:
unmatched = reconciliation[
    reconciliation["_merge"] == "left_only"
]

display(
    unmatched[
        ["Name", "Edition", "Card Number", "Condition", "Language", "Price"]
    ].head(30)
)

,Name,Edition,Card Number,Condition,Language,Price
27,Aegis Angel,Magic Origins,273,Near Mint,English,$0.59
30,Aerathi Berserker,Legends,131,Near Mint,English,$0.56
31,Aether Barrier,Nemesis,27,Near Mint,English,$0.37
32,Aether Mutation,Apocalypse,91,Near Mint,English,$0.23
33,Aether Tide,Exodus,27,Near Mint,English,$0.15
34,Aetherling,Dragon's Maze,11,Near Mint,English,$0.61
35,Aetherspouts,Magic 2015 Core Set,44,Near Mint,English,$0.53
159,Arena,Media Inserts,1,Near Mint,English,$1.34
203,Assault // Battery,Invasion,295,Near Mint,English,$0.18
299,Beck // Call,Dragon's Maze,123,Near Mint,English,$0.62


In [16]:
tt_names = set(trader_tools["match_name"].dropna())
tt_editions = set(trader_tools["match_edition"].dropna())

unmatched = unmatched.copy()

unmatched["name_found"] = unmatched["match_name"].isin(tt_names)
unmatched["edition_found"] = unmatched["match_edition"].isin(tt_editions)

print(
    unmatched.groupby(
        ["name_found", "edition_found"]
    ).size()
)

name_found  edition_found
False       False             2
            True             95
True        False             2
            True             15
dtype: int64


### Exception Classification

The 114 unmatched inventory rows were classified by whether their normalized
card name and edition appeared independently in the Trader Tools dataset.

Results:

- 95 records: edition exists, but card name does not match
- 15 records: both card name and edition exist independently, but not as the same pair
- 2 records: card name exists, but edition does not
- 2 records: neither card name nor edition exists

The largest exception category is therefore card-name reconciliation. This
suggests that differences in source-system naming conventions or formatting
are a more significant cause of failed matches than missing edition data.

In [17]:
name_exceptions = unmatched[
    (~unmatched["name_found"]) &
    (unmatched["edition_found"])
]

display(
    name_exceptions[
        ["Name", "Edition", "Card Number", "Price"]
    ].drop_duplicates(
        subset=["Name", "Edition"]
    ).head(40)
)

,Name,Edition,Card Number,Price
30,Aerathi Berserker,Legends,131,$0.56
31,Aether Barrier,Nemesis,27,$0.37
32,Aether Mutation,Apocalypse,91,$0.23
33,Aether Tide,Exodus,27,$0.15
34,Aetherling,Dragon's Maze,11,$0.61
35,Aetherspouts,Magic 2015 Core Set,44,$0.53
203,Assault // Battery,Invasion,295,$0.18
299,Beck // Call,Dragon's Maze,123,$0.62
427,Brassclaw Orcs,Fallen Empires,100,$0.18
433,Breaking // Entering,Dragon's Maze,124,$0.77


### Observed Name-Reconciliation Patterns

Review of the unmatched card names identified multiple source-system naming issues.

Observed examples include:

- use of `Aether` in TCGplayer versus the `Æther` ligature in Trader Tools;
- split cards represented in TCGplayer using the `Card A // Card B` naming format;
- additional records for which no direct equivalent is yet confirmed in the Trader Tools source.

These exceptions will not be force-matched. Matching rules will only be expanded where a deterministic source-system transformation can be demonstrated.

In [18]:
print("Distinct name exceptions:", len(
    name_exceptions[["Name", "Edition"]].drop_duplicates()
))

print("Split-card exceptions:", (
    name_exceptions[["Name", "Edition"]]
    .drop_duplicates()["Name"]
    .str.contains("//", regex=False)
    .sum()
))

Distinct name exceptions: 50
Split-card exceptions: 16


## 5. Controlled Name Normalization

A deterministic naming difference was identified involving the `Æ` ligature.

Rather than altering source values, the matching fields are standardized so that `Æ` and `Ae` are treated equivalently during reconciliation. Only transformations supported by observed source-system behavior are added.


In [19]:
tcgplayer["match_name_v2"] = (
    tcgplayer["Name"]
    .str.strip()
    .str.lower()
    .str.replace("æ", "ae", regex=False)
)

trader_tools["match_name_v2"] = (
    trader_tools["cardName"]
    .str.strip()
    .str.lower()
    .str.replace("æ", "ae", regex=False)
)

In [20]:
reconciliation_v2 = tcgplayer.merge(
    trader_tools[
        [
            "match_name_v2",
            "match_edition",
            "buyPrice",
            "sellPrice"
        ]
    ],
    on=["match_name_v2", "match_edition"],
    how="left",
    validate="many_to_one",
    indicator=True
)

print(reconciliation_v2["_merge"].value_counts())

_merge
both          4257
left_only      111
right_only       0
Name: count, dtype: int64


### Name-Normalization Improvement

Standardizing the `Æ` ligature to `ae` recovered 3 additional inventory records.

The reconciliation improved from 4,254 matched records to 4,257, reducing unmatched records from 114 to 111.

Because this transformation addresses a known source-system naming difference and can be applied deterministically, it was retained as part of the improved matching logic.

## 6. Split-Card Coverage Investigation

Sixteen unmatched records use TCGplayer's `Card A // Card B` split-card naming convention.

Before changing the matching rule, Trader Tools is inspected to determine whether these records are represented using a different slash convention or are absent from the market dataset entirely.


In [21]:
split_exceptions = reconciliation_v2[
    (reconciliation_v2["_merge"] == "left_only") &
    (reconciliation_v2["Name"].str.contains("//", regex=False))
]

display(
    split_exceptions[
        ["Name", "Edition", "Card Number", "Price"]
    ].drop_duplicates()
)

,Name,Edition,Card Number,Price
203,Assault // Battery,Invasion,295,$0.18
299,Beck // Call,Dragon's Maze,123,$0.62
433,Breaking // Entering,Dragon's Maze,124,$0.77
506,Catch // Release,Dragon's Maze,125,$0.32
1280,Fire // Ice,Apocalypse,128,$0.47
1329,Flesh // Blood,Dragon's Maze,128,$0.29
1846,Illusion // Reality,Apocalypse,129,$0.17
2116,Life // Death,Apocalypse,130,$0.26
2489,Night // Day,Apocalypse,131,$0.18
2593,Order // Chaos,Apocalypse,132,$0.20


In [22]:
trader_tools[
    trader_tools["cardName"]
    .str.contains("Assault|Battery", case=False, na=False)
][["cardName", "setName"]]

,cardName,setName
518,Assault Formation,Dragons of Tarkir
664,Seismic Assault,Exodus
1323,Reckless Assault,Invasion
1557,Black Mana Battery,Legends
1561,Blue Mana Battery,Legends
1600,Green Mana Battery,Legends
1647,Red Mana Battery,Legends
1684,White Mana Battery,Legends
2087,Furious Assault,Mercadian Masques
3970,Relentless Assault,Visions


In [23]:
tt_split_names = trader_tools[
    trader_tools["cardName"].str.contains("/", regex=False, na=False)
]

print("Trader Tools names containing a slash:", len(tt_split_names))
display(tt_split_names[["cardName", "setName"]].head(20))


Trader Tools names containing a slash: 0


,cardName,setName


### Split-Card Exception Classification

Trader Tools contains no card names using a slash-based split-card format, and the sampled search does not identify an alternate representation for `Assault // Battery`.

The 16 unmatched split-card records are therefore treated as **source coverage exceptions**, not as records that should be force-matched through an unsupported normalization rule.

These inventory records remain intact and should be flagged for review when no supported market-data record is available.


## 7. Current Reconstruction State

At this point:

- the source files load successfully;
- `Name + Edition` has been tested as the historical market-data matching concept;
- the relationship validates as many-to-one;
- the first-pass reconciliation matched 4,254 of 4,368 TCGplayer rows;
- deterministic `Æ` → `ae` normalization recovered 3 additional rows;
- the improved reconciliation matches 4,257 rows and leaves 111 unmatched;
- 16 of the remaining unmatched rows are confirmed split-card source-coverage exceptions.

The next step is to classify the remaining unmatched records at a higher level and then compare the reconstructed workflow against the preserved historical Processing output without assuming that `My Price` is a direct copy of Trader Tools `sellPrice`.


## 8. Exception Classification

In [25]:
unmatched_v2 = reconciliation_v2[
    reconciliation_v2["_merge"] == "left_only"
].copy()

tt_names_v2 = set(trader_tools["match_name_v2"].dropna())
tt_editions = set(trader_tools["match_edition"].dropna())

unmatched_v2["name_found"] = unmatched_v2["match_name_v2"].isin(tt_names_v2)
unmatched_v2["edition_found"] = unmatched_v2["match_edition"].isin(tt_editions)

unmatched_v2["exception_reason"] = "Other unmatched record"

unmatched_v2.loc[
    unmatched_v2["Name"].str.contains("//", regex=False, na=False),
    "exception_reason"
] = "Split card absent from Trader Tools"

unmatched_v2.loc[
    (~unmatched_v2["name_found"]) &
    (unmatched_v2["edition_found"]) &
    (~unmatched_v2["Name"].str.contains("//", regex=False, na=False)),
    "exception_reason"
] = "Card name absent from Trader Tools"
    
unmatched_v2.loc[
    (unmatched_v2["name_found"]) &
    (~unmatched_v2["edition_found"]),
    "exception_reason"
] = "Edition absent from Trader Tools"

unmatched_v2.loc[
    (unmatched_v2["name_found"]) &
    (unmatched_v2["edition_found"]),
    "exception_reason"
] = "Name and edition exist, but pair does not match"

unmatched_v2.loc[
    (~unmatched_v2["name_found"]) &
    (~unmatched_v2["edition_found"]),
    "exception_reason"
] = "Name and edition absent from Trader Tools"

print(unmatched_v2["exception_reason"].value_counts())

exception_reason
Card name absent from Trader Tools                 76
Split card absent from Trader Tools                16
Name and edition exist, but pair does not match    15
Edition absent from Trader Tools                    2
Name and edition absent from Trader Tools           2
Name: count, dtype: int64


### Exception Classification Results

The 111 unmatched inventory records were classified as follows:

- 76 — Card name absent from Trader Tools
- 16 — Split card absent from Trader Tools
- 15 — Card name and edition both exist, but not as the same pair
- 2 — Edition absent from Trader Tools
- 2 — Both card name and edition absent from Trader Tools

This confirms that most remaining exceptions are caused by source-system coverage or naming differences rather than merge failure.

In [26]:
output_dir = project_root / "outputs" / "exceptions"
output_dir.mkdir(parents=True, exist_ok=True)

exception_file = output_dir / "reconciliation_exceptions.csv"

unmatched_v2.to_csv(exception_file, index=False)

print("Saved:", exception_file)
print("Rows:", len(unmatched_v2))

Saved: ..\outputs\exceptions\reconciliation_exceptions.csv
Rows: 111


## 9. Historical Processing Comparison

The reconstructed reconciliation will now be compared against the preserved
historical Processing output.

The objective is not to assume that individual pricing fields should be equal.
Instead, the historical output is used to determine how closely the reconstructed
record relationships correspond to the original workbook behavior and to identify
where additional historical business logic may have been applied.

In [31]:
print("Historical Reference columns:")
print(historical_reference.columns.tolist())

print("\nHistorical Reference shape:")
print(historical_reference.shape)

display(historical_reference.head(10))

Historical Reference columns:
['Count', 'Tradelist Count', 'Name', 'Edition', 'My Price', 'Price', 'match_name_v2', 'match_edition']

Historical Reference shape:
(4499, 8)


,Count,Tradelist Count,Name,Edition,My Price,Price,match_name_v2,match_edition
0,1,1,Abbey Matron,Homelands,0.18,$0.18,abbey matron,homelands
1,24,8,Abbot of Keral Keep,Magic Origins,1.63,$2.37,abbot of keral keep,magic origins
2,64,8,Abhorrent Overlord,Theros,0.28,$0.26,abhorrent overlord,theros
3,1,1,Abolish,Prophecy,0.24,$0.24,abolish,prophecy
4,2,2,Abomination,Legends,0.43,$0.45,abomination,legends
5,2,2,About Face,Urza's Legacy,0.18,$0.16,about face,urza's legacy
6,23,8,Abrupt Decay,Return to Ravnica,10.93,$10.93,abrupt decay,return to ravnica
7,1,1,Absorb,Invasion,3.03,$3.09,absorb,invasion
8,2,2,Abyssal Gatekeeper,Weatherlight,0.19,$0.19,abyssal gatekeeper,weatherlight
9,1,1,Abyssal Specter,Classic Sixth Edition,0.24,$0.25,abyssal specter,classic sixth edition


### Historical Record-Structure Comparison

The preserved historical Processing output contains 4,499 rows, compared with
4,368 rows in the TCGplayer inventory input.

Before comparing pricing or business logic, the two datasets are compared at
the normalized `Name + Edition` level to determine whether the difference in
row counts reflects repeated matching relationships, additional historical
records, or another workbook behavior.

In [28]:
historical_reference["match_name_v2"] = (
    historical_reference["Name"]
    .str.strip()
    .str.lower()
    .str.replace("æ", "ae", regex=False)
)

historical_reference["match_edition"] = (
    historical_reference["Edition"]
    .str.strip()
    .str.lower()
)

tcg_key_counts = (
    tcgplayer
    .groupby(["match_name_v2", "match_edition"])
    .size()
    .rename("tcg_rows")
)

historical_key_counts = (
    historical_reference
    .groupby(["match_name_v2", "match_edition"])
    .size()
    .rename("historical_rows")
)

key_comparison = (
    pd.concat(
        [tcg_key_counts, historical_key_counts],
        axis=1
    )
    .fillna(0)
    .astype(int)
)

key_comparison["row_difference"] = (
    key_comparison["historical_rows"]
    - key_comparison["tcg_rows"]
)

print("Keys with equal row counts:",
      (key_comparison["row_difference"] == 0).sum())

print("Keys with more historical rows:",
      (key_comparison["row_difference"] > 0).sum())

print("Keys with more TCGplayer rows:",
      (key_comparison["row_difference"] < 0).sum())

print("Net historical row difference:",
      key_comparison["row_difference"].sum())

display(
    key_comparison[
        key_comparison["row_difference"] != 0
    ]
    .sort_values("row_difference", ascending=False)
    .head(30)
)

Keys with equal row counts: 4120
Keys with more historical rows: 1
Keys with more TCGplayer rows: 0
Net historical row difference: 131


,,tcg_rows,historical_rows,row_difference
match_name_v2,match_edition,,,
0,0,0,131,131


### Historical Row-Count Reconciliation

The historical Processing reference initially contained 4,499 rows compared
with 4,368 TCGplayer inventory rows.

Key-level comparison identified exactly 131 additional historical rows.
Inspection showed that these rows contain zero values across the inventory
fields, including `Name = 0` and `Edition = 0`.

These rows are treated as non-record worksheet residue rather than historical
inventory records.

After excluding the zero-value rows, the historical Processing dataset is
expected to contain the same 4,368 substantive inventory records as the
TCGplayer input.

In [34]:
zero_rows = historical_reference[
    historical_reference["Name"].astype(str).str.strip().eq("0")
    & historical_reference["Edition"].astype(str).str.strip().eq("0")
].copy()

historical_clean = historical_reference[
    ~(
        historical_reference["Name"].astype(str).str.strip().eq("0")
        & historical_reference["Edition"].astype(str).str.strip().eq("0")
    )
].copy()

print("Zero-value worksheet rows:", len(zero_rows))
print("Historical rows after removal:", len(historical_clean))
print("TCGplayer rows:", len(tcgplayer))

Zero-value worksheet rows: 131
Historical rows after removal: 4368
TCGplayer rows: 4368


### Row-Count Validation Result

The 131-row difference between the historical Processing worksheet and the
TCGplayer inventory input was caused entirely by zero-value worksheet residue.

After removing those non-record rows:

- Historical Processing records: 4,368
- TCGplayer inventory records: 4,368

The two datasets therefore contain the same number of substantive inventory
records, allowing direct record-level comparison.

In [35]:
row_identity = (
    tcgplayer["Name"].astype(str).str.strip().eq(
        historical_clean["Name"].astype(str).str.strip().reset_index(drop=True)
    )
    &
    tcgplayer["Edition"].astype(str).str.strip().eq(
        historical_clean["Edition"].astype(str).str.strip().reset_index(drop=True)
    )
)

print("Rows with matching Name + Edition in same position:", row_identity.sum())
print("Total rows:", len(tcgplayer))
print("Rows differing:", (~row_identity).sum())

Rows with matching Name + Edition in same position: 4368
Total rows: 4368
Rows differing: 0


### Row-Order Validation Result

All 4,368 substantive historical Processing records match the TCGplayer
inventory input by `Name + Edition` in the same row position.

This indicates that the historical Processing worksheet preserved the original
inventory record sequence rather than reordering or expanding records during
processing.

Because row order is preserved, the two datasets can now be compared directly
at the record level.

In [37]:
historical_clean = historical_clean.reset_index(drop=True)
tcgplayer_compare = tcgplayer.reset_index(drop=True)

fields_to_check = [
    "Count",
    "Tradelist Count",
    "Name",
    "Edition",
    "Price"
]

for field in fields_to_check:
    matches = (
        tcgplayer_compare[field]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq(
            historical_clean[field]
            .fillna("")
            .astype(str)
            .str.strip()
        )
    )

    print(
        f"{field}: {matches.sum()} matching / "
        f"{len(matches)} total / "
        f"{(~matches).sum()} different"
    )

Count: 4368 matching / 4368 total / 0 different
Tradelist Count: 1479 matching / 4368 total / 2889 different
Name: 4368 matching / 4368 total / 0 different
Edition: 4368 matching / 4368 total / 0 different
Price: 4368 matching / 4368 total / 0 different


### Pass-Through and Tradelist Validation

Record-level comparison confirms that `Count`, `Name`, `Edition`, and `Price`
were preserved unchanged between the TCGplayer inventory input and the
historical Processing worksheet.

`Tradelist Count`, however, differs on 2,889 of 4,368 records, indicating that
it was a calculated output rather than a simple pass-through field.

Historical examples suggest that the workflow limited listed inventory to a
maximum quantity of eight copies per inventory record. This rule is tested
directly against the preserved Processing output before being incorporated
into the reconstruction.

In [38]:
expected_tradelist_count = (
    pd.to_numeric(tcgplayer_compare["Count"], errors="coerce")
    .clip(upper=8)
)

historical_tradelist_count = pd.to_numeric(
    historical_clean["Tradelist Count"],
    errors="coerce"
)

tradelist_rule_matches = expected_tradelist_count.eq(
    historical_tradelist_count
)

print(
    "MIN(Count, 8) matches:",
    tradelist_rule_matches.sum(),
    "/",
    len(tradelist_rule_matches)
)

print(
    "Differences:",
    (~tradelist_rule_matches).sum()
)

MIN(Count, 8) matches: 4368 / 4368
Differences: 0


### Tradelist Count Rule Validation

The historical Processing worksheet recalculated `Tradelist Count` using:

`Tradelist Count = MIN(Count, 8)`

The reconstructed rule matched all 4,368 historical records with zero differences.

This confirms that the workflow limited listed inventory to a maximum of eight
copies per inventory record while preserving the full physical inventory count.

In [39]:
def numeric_price(series):
    return pd.to_numeric(
        series.astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
        .replace({"nan": None, "": None}),
        errors="coerce"
    )

historical_my_price = numeric_price(historical_clean["My Price"])
tcg_original_my_price = numeric_price(tcgplayer_compare["My Price"])
tcg_market_price = numeric_price(tcgplayer_compare["Price"])

recon_compare = reconciliation_v2.reset_index(drop=True)

tt_buy_price = pd.to_numeric(
    recon_compare["buyPrice"],
    errors="coerce"
)

tt_sell_price = pd.to_numeric(
    recon_compare["sellPrice"],
    errors="coerce"
)

print("Historical My Price populated:", historical_my_price.notna().sum())
print("Original TCGplayer My Price populated:", tcg_original_my_price.notna().sum())

print(
    "Historical My Price = original TCGplayer My Price:",
    historical_my_price.eq(tcg_original_my_price).sum()
)

print(
    "Historical My Price = TCGplayer market Price:",
    historical_my_price.eq(tcg_market_price).sum()
)

print(
    "Historical My Price = Trader Tools buyPrice:",
    historical_my_price.eq(tt_buy_price).sum()
)

print(
    "Historical My Price = Trader Tools sellPrice:",
    historical_my_price.eq(tt_sell_price).sum()
)

Historical My Price populated: 4368
Original TCGplayer My Price populated: 1484
Historical My Price = original TCGplayer My Price: 22
Historical My Price = TCGplayer market Price: 462
Historical My Price = Trader Tools buyPrice: 4
Historical My Price = Trader Tools sellPrice: 4253


### Pricing Source Validation

Comparison of the historical `My Price` field against available pricing inputs
shows that Trader Tools `sellPrice` was the primary pricing source for matched
records.

Results:

- Historical `My Price` populated: 4,368 records
- Original TCGplayer `My Price` populated: 1,484 records
- Historical `My Price` matched original TCGplayer `My Price`: 22 records
- Historical `My Price` matched TCGplayer market `Price`: 462 records
- Historical `My Price` matched Trader Tools `buyPrice`: 4 records
- Historical `My Price` matched Trader Tools `sellPrice`: 4,253 records

Of the 4,257 inventory records successfully matched to Trader Tools, 4,253
reproduce the historical `My Price` directly from Trader Tools `sellPrice`.

This strongly supports Trader Tools `sellPrice` as the primary historical pricing
input for matched records while leaving four matched pricing exceptions for
further investigation.

In [40]:
matched_mask = recon_compare["_merge"].eq("both")

pricing_exceptions = pd.DataFrame({
    "Name": tcgplayer_compare["Name"],
    "Edition": tcgplayer_compare["Edition"],
    "TCGplayer Price": tcg_market_price,
    "Historical My Price": historical_my_price,
    "TT buyPrice": tt_buy_price,
    "TT sellPrice": tt_sell_price
})

pricing_exceptions = pricing_exceptions[
    matched_mask
    & historical_my_price.ne(tt_sell_price)
]

print("Matched pricing exceptions:", len(pricing_exceptions))

display(pricing_exceptions)

Matched pricing exceptions: 4


,Name,Edition,TCGplayer Price,Historical My Price,TT buyPrice,TT sellPrice
34,Aetherling,Dragon's Maze,0.61,0.00,NaN,0.43
35,Aetherspouts,Magic 2015 Core Set,0.53,0.00,NaN,0.52
2542,Obscuring Aether,Dragons of Tarkir,0.33,0.00,NaN,0.37
3750,Tarmogoyf,Modern Masters,139.48,274.94,90.0,139.97


### Pricing Exception Analysis

Four records differed between historical `My Price` and the reconstructed
Trader Tools `sellPrice`.

Three of these records — Aetherling, Aetherspouts, and Obscuring Aether —
were only matched after adding the modernized `Æ` / `Ae` normalization rule.

In the historical Processing output, each of these records had a `My Price`
of zero, indicating that the original spreadsheet workflow did not
successfully reconcile them to Trader Tools.

The Python reconstruction therefore improves historical matching for these
records by resolving a deterministic source-system naming difference.

After excluding these three modernization recoveries, only one historically
matched record remains where `My Price` differs from Trader Tools
`sellPrice`: Tarmogoyf from Modern Masters.

This record requires separate investigation to determine whether the
difference resulted from an inventory attribute, manual override, or
additional historical pricing rule.

In [41]:
tarmogoyf_index = pricing_exceptions[
    pricing_exceptions["Name"] == "Tarmogoyf"
].index[0]

display(
    tcgplayer_compare.loc[
        [tarmogoyf_index],
        [
            "Count",
            "Tradelist Count",
            "Name",
            "Edition",
            "Card Number",
            "Condition",
            "Language",
            "Foil",
            "My Price",
            "Price",
            "Rarity"
        ]
    ]
)

print("Trader Tools buyPrice:", tt_buy_price.loc[tarmogoyf_index])
print("Trader Tools sellPrice:", tt_sell_price.loc[tarmogoyf_index])
print("Historical My Price:", historical_my_price.loc[tarmogoyf_index])

,Count,Tradelist Count,Name,Edition,Card Number,Condition,Language,Foil,My Price,Price,Rarity
3750,1,1,Tarmogoyf,Modern Masters,166,Near Mint,English,NaN,28150.0,$139.48,MythicRare


Trader Tools buyPrice: 90.0
Trader Tools sellPrice: 139.97
Historical My Price: 274.94


In [42]:
print("TCGplayer Tarmogoyf records:")
display(
    tcgplayer_compare[
        tcgplayer_compare["Name"].str.contains(
            "Tarmogoyf",
            case=False,
            na=False
        )
    ]
)

print("\nTrader Tools Tarmogoyf records:")
display(
    trader_tools[
        trader_tools["cardName"].str.contains(
            "Tarmogoyf",
            case=False,
            na=False
        )
    ]
)

print("\nHistorical Tarmogoyf records:")
display(
    historical_clean[
        historical_clean["Name"].str.contains(
            "Tarmogoyf",
            case=False,
            na=False
        )
    ]
)

TCGplayer Tarmogoyf records:


,Count,Tradelist Count,Name,Edition,Card Number,Condition,Language,Foil,Signed,Artist Proof,Altered Art,Misprint,Promo,Textless,My Price,Rarity,Price,match_name,match_edition,match_name_v2
3750,1,1,Tarmogoyf,Modern Masters,166,Near Mint,English,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28150.0,MythicRare,$139.48,tarmogoyf,modern masters,tarmogoyf
3751,2,2,Tarmogoyf,Modern Masters 2015 Edition,165,Near Mint,English,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13650.0,MythicRare,$134.92,tarmogoyf,modern masters 2015 edition,tarmogoyf
3752,1,1,Tarmogoyf,Future Sight,153,Near Mint,English,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16975.0,Rare,$168.93,tarmogoyf,future sight,tarmogoyf



Trader Tools Tarmogoyf records:


,cardName,setName,foil,qtyOwned,buyPrice,sellPrice,buyMerchant,sellMerchant,buyQty,sellQty,spread,match_name,match_edition,match_name_v2
886,Tarmogoyf,Future Sight,0,1.0,114.90,164.15,isleofcards,isleofcards,6.0,4.0,%,tarmogoyf,future sight,tarmogoyf
2415,Tarmogoyf,Modern Masters,0,1.0,90.00,139.97,magicdotcards,tcgplayer,NaN,NaN,%,tarmogoyf,modern masters,tarmogoyf
2424,Tarmogoyf,Modern Masters 2015 Edition,0,2.0,83.26,134.97,abugames,tcgplayer,12.0,NaN,%,tarmogoyf,modern masters 2015 edition,tarmogoyf



Historical Tarmogoyf records:


,Count,Tradelist Count,Name,Edition,My Price,Price,match_name_v2,match_edition
3750,1,1,Tarmogoyf,Modern Masters,274.94,$139.48,tarmogoyf,modern masters
3751,2,2,Tarmogoyf,Modern Masters 2015 Edition,134.97,$134.92,tarmogoyf,modern masters 2015 edition
3752,1,1,Tarmogoyf,Future Sight,164.15,$168.93,tarmogoyf,future sight


### Historical Lookup Limitation

The original spreadsheet workflow prioritized rapid operational processing over
eliminating every possible matching exception.

Fresh TCGplayer and Trader Tools data could be copied into the workbook with
minimal manual effort, allowing thousands of inventory records to be repriced
and prepared for export quickly.

A small number of known edge cases, such as Tarmogoyf, could produce incorrect
results when spreadsheet lookup behavior more than one edition price contributing to the result. Because the overall workflow was inexpensive to run and these exceptions
were relatively easy to identify and correct manually, this limitation was
operationally acceptable at the time.

The Python modernization does not invalidate that original design decision.
Instead, it improves the workflow by making matching rules explicit, validating
relationships before joining data, and automatically surfacing exceptions that
previously depended on manual review.

In [43]:
display(
    trader_tools[
        trader_tools["cardName"].str.contains(
            "Tarmogoyf",
            case=False,
            na=False
        )
    ][
        ["cardName", "setName", "buyPrice", "sellPrice"]
    ]
)

,cardName,setName,buyPrice,sellPrice
886,Tarmogoyf,Future Sight,114.90,164.15
2415,Tarmogoyf,Modern Masters,90.00,139.97
2424,Tarmogoyf,Modern Masters 2015 Edition,83.26,134.97


### Tarmogoyf Historical Lookup Defect

The historical Tarmogoyf pricing anomaly was reconstructed from the preserved
Trader Tools data.

For Tarmogoyf, Trader Tools contained:

- Future Sight: `sellPrice = 164.15`
- Modern Masters: `sellPrice = 139.97`
- Modern Masters 2015 Edition: `sellPrice = 134.97`

The historical Processing worksheet assigned the Modern Masters record a
`My Price` of `274.94`.

This value exactly equals:

`139.97 + 134.97 = 274.94`

The historical workbook therefore appears to have matched both `Modern Masters`
and `Modern Masters 2015 Edition` when calculating the price for the Modern
Masters printing.

This indicates that the original spreadsheet matching logic allowed an edition
name to match another edition containing the same text rather than enforcing an
exact card-name-and-edition relationship.

The Python reconstruction prevents this defect by using explicit normalized
`Name + Edition` keys and validating the relationship as many-to-one before
performing the merge.

In [44]:
tarmogoyf_masters = trader_tools[
    (trader_tools["cardName"] == "Tarmogoyf")
    & (
        trader_tools["setName"].isin([
            "Modern Masters",
            "Modern Masters 2015 Edition"
        ])
    )
].copy()

masters_sellprice_sum = pd.to_numeric(
    tarmogoyf_masters["sellPrice"],
    errors="coerce"
).sum()

historical_tarmogoyf_price = historical_clean.loc[
    historical_clean["Name"].eq("Tarmogoyf")
    & historical_clean["Edition"].eq("Modern Masters"),
    "My Price"
].iloc[0]

print("Modern Masters sellPrice sum:", masters_sellprice_sum)
print("Historical Modern Masters My Price:", historical_tarmogoyf_price)
print(
    "Exact match:",
    round(masters_sellprice_sum, 2)
    == round(float(historical_tarmogoyf_price), 2)
)

Modern Masters sellPrice sum: 274.94
Historical Modern Masters My Price: 274.94
Exact match: True


### Tarmogoyf Defect Validation Result

The historical Tarmogoyf anomaly was reproduced directly from the preserved
Trader Tools data.

The Trader Tools `sellPrice` values were:

- Modern Masters: $139.97
- Modern Masters 2015 Edition: $134.97

Their combined value is:

`$139.97 + $134.97 = $274.94`

The historical Processing worksheet assigned the Modern Masters Tarmogoyf a
`My Price` of exactly $274.94.

This demonstrates that both Masters-edition price records contributed to the
historical result rather than the intended Modern Masters record being isolated.

The surviving data therefore confirms the historical matching defect, although
the precise spreadsheet formula responsible is not inferred without direct
formula evidence.

The Python reconstruction prevents this condition by matching on explicit,
normalized `Name + Edition` keys and validating the market-data relationship
before merging.

## 11. Modernized Reconciliation Output

The validated historical business rules can now be implemented as an explicit
Python workflow.

The modernized output will:

- preserve the original TCGplayer inventory records;
- calculate `Tradelist Count` as `MIN(Count, 8)`;
- use Trader Tools `sellPrice` when an exact normalized `Name + Edition` match exists;
- retain the original source fields for traceability; and
- flag unmatched records for review rather than silently assigning unsupported prices.

In [45]:
modernized_output = tcgplayer_compare.copy()

modernized_output["Reconstructed Tradelist Count"] = (
    pd.to_numeric(
        modernized_output["Count"],
        errors="coerce"
    ).clip(upper=8)
)

modernized_output["Trader Tools Sell Price"] = tt_sell_price

modernized_output["Reconstructed My Price"] = tt_sell_price

modernized_output["Reconciliation Status"] = (
    recon_compare["_merge"]
    .map({
        "both": "Matched",
        "left_only": "Exception"
    })
    .astype("object")
)

print(
    modernized_output["Reconciliation Status"]
    .value_counts()
)

display(
    modernized_output[
        [
            "Name",
            "Edition",
            "Count",
            "Reconstructed Tradelist Count",
            "Price",
            "Trader Tools Sell Price",
            "Reconstructed My Price",
            "Reconciliation Status"
        ]
    ].head(10)
)

Reconciliation Status
Matched      4257
Exception     111
Name: count, dtype: int64


,Name,Edition,Count,Reconstructed Tradelist Count,Price,Trader Tools Sell Price,Reconstructed My Price,Reconciliation Status
0,Abbey Matron,Homelands,1,1,$0.18,0.18,0.18,Matched
1,Abbot of Keral Keep,Magic Origins,24,8,$2.37,1.63,1.63,Matched
2,Abhorrent Overlord,Theros,64,8,$0.26,0.28,0.28,Matched
3,Abolish,Prophecy,1,1,$0.24,0.24,0.24,Matched
4,Abomination,Legends,2,2,$0.45,0.43,0.43,Matched
5,About Face,Urza's Legacy,2,2,$0.16,0.18,0.18,Matched
6,Abrupt Decay,Return to Ravnica,23,8,$10.93,10.93,10.93,Matched
7,Absorb,Invasion,1,1,$3.09,3.03,3.03,Matched
8,Abyssal Gatekeeper,Weatherlight,2,2,$0.19,0.19,0.19,Matched
9,Abyssal Specter,Classic Sixth Edition,1,1,$0.25,0.24,0.24,Matched


### Exception Integration

The exception classifications developed during reconciliation are now incorporated
into the modernized output.

Matched records require no exception reason. Unmatched records retain their
specific classification so that failed reconciliations can be reviewed without
losing the original inventory record.

In [46]:
modernized_output["Exception Reason"] = None

modernized_output.loc[
    unmatched_v2.index,
    "Exception Reason"
] = unmatched_v2["exception_reason"]

print(
    modernized_output[
        modernized_output["Reconciliation Status"] == "Exception"
    ]["Exception Reason"].value_counts()
)

Exception Reason
Card name absent from Trader Tools                 76
Split card absent from Trader Tools                16
Name and edition exist, but pair does not match    15
Edition absent from Trader Tools                    2
Name and edition absent from Trader Tools           2
Name: count, dtype: int64


### Modernized Reconciliation Export

The completed reconciliation output is exported as an auditable project artifact.

The export preserves the original TCGplayer inventory fields alongside the
reconstructed values, Trader Tools market price, reconciliation status, and
exception reason.

This file is intended for validation and portfolio evidence. A separate
TCGplayer-compatible operational export can be produced after the reconstructed
output has been validated.

In [47]:
reconciliation_output_dir = project_root / "outputs" / "reconciliation"
reconciliation_output_dir.mkdir(parents=True, exist_ok=True)

modernized_export_file = (
    reconciliation_output_dir
    / "modernized_reconciliation_output.csv"
)

export_columns = [
    "Count",
    "Tradelist Count",
    "Name",
    "Edition",
    "Card Number",
    "Condition",
    "Language",
    "Foil",
    "Signed",
    "Artist Proof",
    "Altered Art",
    "Misprint",
    "Promo",
    "Textless",
    "My Price",
    "Rarity",
    "Price",
    "Reconstructed Tradelist Count",
    "Trader Tools Sell Price",
    "Reconstructed My Price",
    "Reconciliation Status",
    "Exception Reason"
]

modernized_output[
    export_columns
].to_csv(
    modernized_export_file,
    index=False
)

print("Saved:", modernized_export_file)
print("Rows exported:", len(modernized_output))
print()
print(modernized_output["Reconciliation Status"].value_counts())

Saved: ..\outputs\reconciliation\modernized_reconciliation_output.csv
Rows exported: 4368

Reconciliation Status
Matched      4257
Exception     111
Name: count, dtype: int64


## 12. Modernized Output Validation

The exported reconciliation dataset is reloaded and independently validated to
confirm that the modernization preserves the original inventory records and
applies the reconstructed business rules correctly.

Validation checks include:

- preservation of all 4,368 source inventory records;
- preservation of source `Name`, `Edition`, `Count`, and market `Price`;
- correct application of `Tradelist Count = MIN(Count, 8)`;
- Trader Tools `sellPrice` used only for successfully matched records;
- unmatched records retained and explicitly flagged as exceptions.

In [48]:
validated_output = pd.read_csv(modernized_export_file)

print("Exported rows:", len(validated_output))

print(
    "Matched records:",
    (validated_output["Reconciliation Status"] == "Matched").sum()
)

print(
    "Exception records:",
    (validated_output["Reconciliation Status"] == "Exception").sum()
)

source_name_match = (
    validated_output["Name"].astype(str)
    == tcgplayer_compare["Name"].astype(str)
)

source_edition_match = (
    validated_output["Edition"].astype(str)
    == tcgplayer_compare["Edition"].astype(str)
)

source_count_match = (
    pd.to_numeric(validated_output["Count"], errors="coerce")
    == pd.to_numeric(tcgplayer_compare["Count"], errors="coerce")
)

tradelist_rule_match = (
    pd.to_numeric(
        validated_output["Reconstructed Tradelist Count"],
        errors="coerce"
    )
    ==
    pd.to_numeric(
        validated_output["Count"],
        errors="coerce"
    ).clip(upper=8)
)

matched_price_check = (
    validated_output.loc[
        validated_output["Reconciliation Status"] == "Matched",
        "Reconstructed My Price"
    ]
    ==
    validated_output.loc[
        validated_output["Reconciliation Status"] == "Matched",
        "Trader Tools Sell Price"
    ]
)

print("Name preserved:", source_name_match.all())
print("Edition preserved:", source_edition_match.all())
print("Count preserved:", source_count_match.all())
print("Tradelist rule valid:", tradelist_rule_match.all())
print("Matched pricing valid:", matched_price_check.all())

Exported rows: 4368
Matched records: 4257
Exception records: 111
Name preserved: True
Edition preserved: True
Count preserved: True
Tradelist rule valid: True
Matched pricing valid: True


### Modernized Output Validation Result

The exported modernized reconciliation output passed all defined validation checks.

Results:

- 4,368 source inventory records preserved
- 4,257 records successfully reconciled to Trader Tools
- 111 records retained as explicit reconciliation exceptions
- `Name` preserved for all records
- `Edition` preserved for all records
- `Count` preserved for all records
- `Tradelist Count = MIN(Count, 8)` validated for all records
- matched `Reconstructed My Price` values equal Trader Tools `sellPrice`

The reconstruction also identified and corrected two historical limitations:

- deterministic `Ae` / `Æ` normalization recovered three records missed by the original spreadsheet workflow;
- exact `Name + Edition` matching prevents the historical Tarmogoyf defect in which both Modern Masters pricing records contributed to a single result.

The reconstructed workflow therefore preserves the operational behavior of the
historical system while adding explicit validation, deterministic matching, and
auditable exception handling.